In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
RAW=ROOT / "data" / "raw" / ("PRSA_Data_Aotizhongxin_20130301-20170228.csv")
OUT = ROOT / "data" / "processed"
REPORTS =ROOT / "reports"
OUT.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
if not RAW.is_file():
    raise FileNotFoundError(f"CSV not found: {RAW}")



In [3]:
df=pd.read_csv(RAW)
print("Rows, columns:", df.shape)
print("Column names:", df.columns.tolist())

Rows, columns: (35064, 18)
Column names: ['No', 'year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM', 'station']


In [ ]:
time_parts = df[["year","month","day","hour"]]
df["timestamp"]=pd.to_datetime(time_parts,errors="coerce")

print("Invalid Timestamps:",df["timestamp"].isna().sum())
print("Repeated Timestamps:",df["timestamp"].duplicated().sum())
print("Already Chronological:",df["timestamp"].is_monotonic_increasing)

Invalid Timestamps: 0
Repeated Timestamps: 0
Already Chronological: True


### Measure missingness and duplicates

In [6]:
missing_count=df.isna().sum().sort_values(ascending=False)
missing_percent= (df.isna().mean()*100).round(2)
exact_duplicates = df.duplicated().sum()

print("Missing values by columns:")
print(missing_count)
print("Missing Percentage:")
print(missing_percent)
print("Exact duplicated rows:", exact_duplicates)

Missing values by columns:
CO           1776
O3           1719
NO2          1023
SO2           935
PM2.5         925
PM10          718
wd             81
RAIN           20
PRES           20
DEWP           20
TEMP           20
WSPM           14
month           0
No              0
year            0
hour            0
day             0
station         0
timestamp       0
dtype: int64
Missing Percentage:
No           0.00
year         0.00
month        0.00
day          0.00
hour         0.00
PM2.5        2.64
PM10         2.05
SO2          2.67
NO2          2.92
CO           5.07
O3           4.90
TEMP         0.06
PRES         0.06
DEWP         0.06
RAIN         0.06
wd           0.23
WSPM         0.04
station      0.00
timestamp    0.00
dtype: float64
Exact duplicated rows: 0


### Flag physically invalid pollutant measurements 

In [7]:
pollutants = ["PM2.5","PM10","SO2","NO2","CO","O3"]
negative_counts = {}
for name in pollutants:
    if name in df.columns:
        negative_counts[name] = (df[name]<0).sum()
print("Negative values by columns:")
print(negative_counts)


Negative values by columns:
{'PM2.5': np.int64(0), 'PM10': np.int64(0), 'SO2': np.int64(0), 'NO2': np.int64(0), 'CO': np.int64(0), 'O3': np.int64(0)}


#### Apply conservation, documented cleaning rules

In [8]:
clean = df.drop_duplicates().copy()
clean = clean.dropna(subset=["timestamp"]).copy()
clean = clean.sort_values("timestamp").reset_index(drop=True)

for name in pollutants:
    if name in clean.columns:
        clean.loc[clean[name]<0, name] = float("nan")
remaining_repeated_times = int(clean["timestamp"].duplicated().sum())
print("Repeated timesstamps after cleaning:", remaining_repeated_times)
        

Repeated timesstamps after cleaning: 0


#### Check fir missing hours without investing measurements

In [9]:
unique_times = pd.DatetimeIndex(clean["timestamp"].dropna().unique())
if len(unique_times) > 0:
    expected_hours = pd.date_range(
        start=unique_times.min(), end=unique_times.max(), freq="h"
    )
    missing_hours = expected_hours.difference(unique_times)
else:
    missing_hours = pd.DatetimeIndex([])
print("Missing hourly timestamps:", len(missing_hours))

Missing hourly timestamps: 0


#### Save cleaned data and a reproducible report

In [10]:
clean_path = OUT / "aotizhongxin_quality_checked.csv"
report_path = REPORTS / "practical03_quality_report.txt"
clean.to_csv(clean_path, index=False)
lines = [
    "Practical 03: Aotizhongxin Data Quality Report",
    f"Original rows: {len(df)}",
    f"Cleaned rows: {len(clean)}",
    f"Exact duplicate rows in original: {exact_duplicates}",
    f"Invalid original timestamps: {df['timestamp'].isna().sum()}",
    f"Repeated timestamps remaining: {remaining_repeated_times}",
    f"Missing hourly timestamps: {len(missing_hours)}",
    "", "Original missing counts:", missing_count.to_string(),
    "", "Negative pollutant counts:", str(negative_counts),
    "", "Cleaning policy: no missing-value imputation performed.",
]
report_path.write_text("\n".join(lines), encoding="utf-8")
print("Saved:", clean_path)
print("Saved:", report_path)

Saved: c:\Users\Lone Anees\Desktop\Beijing_MLOps\data\processed\aotizhongxin_quality_checked.csv
Saved: c:\Users\Lone Anees\Desktop\Beijing_MLOps\reports\practical03_quality_report.txt
